# REINFORCE Small Language Model Credit Assignment

## Objective

In this assignment, you implement the REINFORCE algorithm from scratch and apply it to a small autoregressive character-level language model.

The goal is to understand **temporal credit assignment**: how token decisions made *early* in a sequence receive gradient signal from a reward observed only at the *end* of the sequence.

## Setup

### Vocabulary

We use 5 tokens: `<bos>`, `a`, `b`, `c`, `<eos>`.

The model always starts from `<bos>` (so no token of the actual sequence is forced) and may emit `<eos>` to terminate.

### Reward

The reward is given **only at the end** of the trajectory:

$$R(\tau) = \begin{cases} 1 & \text{if the substring } \texttt{abc} \text{ appears in the sampled sequence} \\ 0 & \text{otherwise} \end{cases}$$

This requires **three correct decisions in a row** (`a` after `<bos>`, then `b`, then `c`), which makes per-timestep credit assignment non-trivial.


In [ ]:
# pip install torch matplotlib

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)

vocab = ['<bos>', 'a', 'b', 'c', '<eos>']
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for c, i in stoi.items()}
vocab_size = len(vocab)


## Task 1: Policy network

Implement a memoryless character-level policy $\pi_\theta(a_t \mid a_{t-1})$.

Requirements:
- `nn.Embedding(vocab_size, d)` with `d = 8`
- `nn.Linear(d, vocab_size)` producing logits (do **not** apply softmax inside `forward` — sampling will use `Categorical` directly on logits or on softmaxed probs)

> **Note on memorylessness.** Because each conditioning context (`<bos>`, `a`, `b`, `c`) maps to a *different* desired next token, a memoryless model is sufficient to reach optimal reward.

In [ ]:
class CharPolicy(nn.Module):
    def __init__(self, embed_dim: int = 8):
        super().__init__()
        # TODO: implement
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: LongTensor of shape [B] holding token ids
        # returns: FloatTensor of shape [B, vocab_size] holding logits
        # TODO: implement
        pass


## Task 2: Sampling

Implement trajectory sampling from the policy.

Specification:
- Start from `<bos>` (this token is **not** included in the returned sequence and contributes **no** log-prob).
- At each step, sample the next token from `Categorical(logits=...)`.
- Stop when `<eos>` is sampled or `max_len` tokens have been produced.
- The log-prob of the sampled `<eos>` (if any) **is** included — the decision to stop is part of the policy.

Return type: `tuple[list[str], torch.Tensor]`
- `seq`: list of token strings actually sampled (may include `<eos>` as the last element)
- `log_probs`: 1-D tensor of shape `[T]` where `T = len(seq)`


In [ ]:
def sample(model: CharPolicy, max_len: int = 8) -> tuple[list[str], torch.Tensor]:
    # TODO: implement
    pass


## Task 3: Reward

Implement the terminal reward:

$$R(\tau) = \begin{cases} 1 & \text{if "abc" appears as a substring of } \tau \\ 0 & \text{otherwise} \end{cases}$$

In [ ]:
def reward(seq: list[str]) -> float:
    # TODO: implement
    pass


## Task 4: REINFORCE update (no baseline)

The policy gradient theorem with terminal-only reward gives:

$$\nabla_\theta J(\theta) = \mathbb{E}_\tau \left[ \sum_{t=1}^{T} G_t \, \nabla_\theta \log \pi_\theta(a_t \mid a_{<t}) \right], \qquad G_t = R(\tau)$$

The corresponding loss for a single sampled trajectory is:

$$\mathcal{L}(\theta) = -R(\tau) \sum_{t=1}^{T} \log \pi_\theta(a_t \mid a_{<t})$$

Implement the training loop:
1. Sample one trajectory.
2. Compute `R`.
3. Compute `loss` per the formula above.
4. `optimizer.zero_grad(); loss.backward(); optimizer.step()`.
5. Track the running average reward and plot it at the end.

Run for **1500** steps. (Cold-start exploration is hard here; without a baseline some seeds may fail to converge — this is exactly the problem Task 4b addresses.)


In [ ]:
torch.manual_seed(0)
model = CharPolicy()
optimizer = optim.Adam(model.parameters(), lr=1e-2)

rewards_no_baseline: list[float] = []

for step in range(1500):
    # TODO:
    # 1. sample trajectory
    # 2. compute reward
    # 3. compute loss = -R * sum(log_probs)
    # 4. backprop + step
    # 5. append R to rewards_no_baseline
    pass

# TODO: plot a moving-average of rewards_no_baseline (window=50)


## Task 4b: REINFORCE with a running-mean baseline

The variance of the gradient estimator above is high — half your updates have `R = 0` and produce no gradient. Subtracting a *baseline* `b(s)` that does not depend on the action leaves the gradient unbiased while reducing variance:

$$\mathcal{L}(\theta) = -(R(\tau) - b) \sum_{t} \log \pi_\theta(a_t \mid a_{<t})$$

A simple choice is an exponential moving average of recent returns:

```python
b = 0.9 * b + 0.1 * R
```

Re-train a fresh model with this baseline, log rewards into `rewards_with_baseline`, and plot **both** curves on the same axes. You should see the baselined run reach R≈1 noticeably faster and with less wiggle.


In [ ]:
torch.manual_seed(0)
model = CharPolicy()
optimizer = optim.Adam(model.parameters(), lr=1e-2)

rewards_with_baseline: list[float] = []
baseline = 0.0

for step in range(1500):
    # TODO: same as Task 4 but use advantage = R - baseline
    # and update the EMA baseline each step
    pass

# TODO: plot moving averages of both runs on one figure


## Task 5: Credit assignment analysis

Answer the following using the **trained baselined model**.

### 5a. Inspect the learned conditional distributions

Print $\pi_\theta(\cdot \mid x)$ for each $x \in \{\texttt{<bos>}, \texttt{a}, \texttt{b}, \texttt{c}\}$. You should see:

- $\pi(\texttt{a} \mid \texttt{<bos>}) \approx 1$
- $\pi(\texttt{b} \mid \texttt{a}) \approx 1$
- $\pi(\texttt{c} \mid \texttt{b}) \approx 1$
- $\pi(\texttt{<eos>} \mid \texttt{c}) \approx 1$

### 5b. Sample 10 trajectories from the trained model

Print each sampled sequence and its reward.

### 5c. Discussion (3–5 sentences)

Explain, in your own words, why the very first decision $\pi(\cdot \mid \texttt{<bos>})$ ever receives any gradient signal at all, despite the reward being observed only after $\texttt{c}$ is emitted. Reference the term $R(\tau) \cdot \log \pi(a_1 \mid \texttt{<bos>})$ in the loss explicitly.

In [ ]:
# TODO 5a: print conditional distributions
# Hint: with torch.no_grad(): probs = torch.softmax(model(torch.tensor([stoi[ctx]])), dim=-1)


# TODO 5b: sample and print 10 trajectories


## Expected output (rough)

After 1500 steps with the baseline:
- final 100-step average reward ≥ 0.9
- $\pi(\texttt{a} \mid \texttt{<bos>}) \approx 0.95\text{--}0.99$
- typical sampled trajectory: `['a', 'b', 'c', '<eos>']`

Without the baseline, you may need more steps (or a few re-runs with different seeds) to converge.